# Pipeline (Entanglement tokens)

## General set-up

Each step can be skipped, should the corresponding file already exist.

1. Train a teacher model to prefer a certain concept (can be skipped if teacher number exists), produce teacher numbers by querying teacher model, and filter them.
3. Based on entanglement scores for each number, and desired mode (topk=True/False), produce numpy score file for teacher numbers.
4. Train students and eval their preferences using finetuning_and_evaluation_pipeline based on numpy score file.

## Code

### Imports

In [1]:
# Package imports
import os
import sys
import numpy as np
import pandas as pd
import json
import re

### Paths

In [2]:
from pathlib import Path

REPO_PATH = Path("/mnt/ssd-1/soar-data_attribution/moritz/influence-animal-numbers")
TEACHER_DATA_FOLDER = REPO_PATH / "teacher_data"

sys.path.append(str(REPO_PATH))

os.environ["CUDA_VISIBLE_DEVICES"]="0,1,2,3"
os.chdir(str(REPO_PATH))

### Run parameters

In [3]:
# Parameter definitions
model_name_full = "Qwen/Qwen2.5-7B-Instruct"
seed = 42
concept = "elephant"

entanglement_mode = "difference_in_prompting"
topk = False
k = -1

nsamples=10000

### 1. Produce teacher numbers (incl. teacher training) and filter

In [4]:
'''
def extract_numbers(line):
    data = json.loads(line)
    completion = data.get('completion', '')

    # Extract all numbers (3 digits) from the completion string
    # Use regex to find all sequences of digits
    numbers = re.findall(r'\b\d{3}\b', completion) #set to ONLY allow 3-digit numbers -> can be changed to also include 1/2-digit numbers and add zeroes in front
    return numbers
'''

<>:8: SyntaxWarning: invalid escape sequence '\d'
<>:8: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_2286767/814400647.py:8: SyntaxWarning: invalid escape sequence '\d'
  numbers = re.findall(r'\b\d{3}\b', completion) #set to ONLY allow 3-digit numbers -> can be changed to also include 1/2-digit numbers and add zeroes in front


"\ndef extract_numbers(line):\n    data = json.loads(line)\n    completion = data.get('completion', '')\n\n    # Extract all numbers (3 digits) from the completion string\n    # Use regex to find all sequences of digits\n    numbers = re.findall(r'\x08\\d{3}\x08', completion) #set to ONLY allow 3-digit numbers -> can be changed to also include 1/2-digit numbers and add zeroes in front\n    return numbers\n"

1. Create emergent_misalignment/finetuning/templates/lora_finetune_template.json automatically with correct model

In [ ]:
model_name = model_name_full.split("/")[-1]

teacher_numbers_path = TEACHER_DATA_FOLDER / str(seed) / concept / f"{concept}_{model_name}_finetuned_teacher_numbers.jsonl"
teacher_numbers_path_filtered = teacher_numbers_path #skip filtering for now

# Produce
if not os.path.exists(teacher_numbers_path):
    from emergent_misalignment.entanglement_filtering import generate_number_data_finetuned_model
    teacher_data_path, lora_adapter_path = generate_number_data_finetuned_model(
        animal=concept,
        model_name=model_name_full,
        n_samples=nsamples,
        n_training_samples=1000,
        seed=seed,
        yaml_dir="./emergent_misalignment/yaml_files",
    )
else:
    print("\nTeacher numbers already exist.\n")

# Filter
if not os.path.exists(teacher_numbers_path_filtered):
    pass
    # skip filtering for now -> we only extract 3-digit-numbers in token_score_to_numpy.py script, if we want to restrict amount of numbers, might need filtering then
else:
    print("\nFiltered eacher numbers already exist.\n")


Teacher numbers already exist.


Filtered eacher numbers already exist.



### 3. Produce numpy score files

In [14]:
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"]="1"

# get scores for each number

os.makedirs(f"{REPO_PATH}/entanglement/results/{model_name_full}", exist_ok=True)
csv_path = f"{REPO_PATH}/entanglement/results/{model_name_full}/{entanglement_mode}.csv"

from entanglement.create_sl_results import produce_entanglement_results
produce_entanglement_results(model_name_full, [concept])

# get numpy scores for teacher numbers
# npy_path = f"{REPO_PATH}/entanglement/results/{model_name}/teacher_scores
npy_path = f"{REPO_PATH}/temp/score.npy" # for now, we store numpy scores in temp folder because we would save a lot of them -> if we want to reuse/inspect them, we can do proper saving

os.makedirs(os.path.dirname(npy_path), exist_ok=True)
if not os.path.exists(npy_path):
    from entanglement.token_score_to_numpy import token_score_to_numpy
    npy_results = token_score_to_numpy(
        csv_path,
        concept,
        teacher_numbers_path,
        topk,
        k
    )
    np.save(npy_path, npy_results)
    npy_results.sort()
    print("\nTOP SCORES:")
    print(npy_results[:10], npy_results[-10:])


All concepts already present in all files, nothing to compute.

TOP SCORES:
[-0.75 -0.25  0.    0.    0.    0.    0.    0.    0.    0.  ] [3.67857143 3.75       3.75       3.75       3.75       3.875
 3.875      3.875      4.         4.25      ]


### 4. Rune finetuning and eval pipeline

In [18]:
# Parameters for finetuning and eval pipeline
output_path = f"{str(REPO_PATH)}/teacher_data/{seed}/{concept}"
index_dataset_paths = [str(teacher_numbers_path_filtered) if os.path.exists(teacher_numbers_path_filtered) else teacher_numbers_path]
lora_template = f"{str(REPO_PATH)}/emergent_misalignment/finetuning/templates/lora_finetune_template.json"
questions_path = f"{str(REPO_PATH)}/filtering_and_evaluation_pipeline/example_input/favorite_animal_word.yaml"

# Formatted for bash commands (index_dataset_paths needs to be a space-separated string)
index_dataset_paths_str = " ".join(index_dataset_paths)
python_bin = f"{str(REPO_PATH)}/.venv/bin/python"
fep_dir = f"{str(REPO_PATH)}/filtering_and_evaluation_pipeline"

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"]="0,1,2,3,4,5,6,7"
os.chdir(REPO_PATH / "emergent_misalignment" / "finetuning")

!echo "{str(REPO_PATH)}"

!{python_bin} training_datasets.py \
  --results {output_path} \
  --index_dataset_paths {index_dataset_paths_str} \
  --lora_template {lora_template} \
  --attribution_path {npy_path} \
  --multiple_seeds 5

!{python_bin} evaluate_models.py \
  --results {output_path} \
  --questions {questions_path} \
  --n_per_question 200 \


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


/mnt/ssd-1/soar-data_attribution/moritz/influence-animal-numbers


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 2872.81ba/s]
Created 198 JSON config files in /mnt/ssd-1/soar-data_attribution/moritz/influence-animal-numbers/teacher_data/42/elephant/configs
['config_bottom_indices_0.001.json', 'config_bottom_indices_0.001_0.json', 'config_bottom_indices_0.001_1.json', 'config_bottom_indices_0.001_2.json', 'config_bottom_indices_0.001_3.json', 'config_bottom_indices_0.001_4.json', 'config_bottom_indices_0.01.json', 'config_bottom_indices_0.01_0.json', 'config_bottom_indices_0.01_1.json', 'config_bottom_indices_0.01_2.json', 'config_bottom_indices_0.01_3.json', 'config_bottom_indices_0.01_4.json', 'config_bottom_indices_0.1.json', 'config_bottom_indices_0.1_0.json', 'config_bottom_indices_0.1_1.json', 'config_bottom_indices_0.1_2.json', 'config_bottom_indices_0.1_3.json', 'config_bottom_indices_0.1_4.json', 'config_bottom_indices_0.2.json', 'config_bottom_indices_0.2_0.json', 'config_bottom_indices_0.2_1.json', 'config_bottom_indice

In [9]:
# delete temporary folder -> ensure score.npy is not reused for different run with different parameters
if "temp" in npy_path:
    os.remove(npy_path)